# Chapter 6: AI Agents in Retail and E-Commerce

## Complete Code Implementations

This notebook contains all code listings from Chapter 6, covering AI agents for retail and e-commerce applications including:

- **6.1** Core Retail Agent State Schema
- **6.2** Agentic RAG for Product Knowledge
- **6.3** Hybrid Personalization Engine
- **6.4** Multi-Modal Service Agent
- **6.5** Omnichannel Adapters
- **6.6** Intelligent Returns Processing Agent
- **6.7** Fraud Investigation Agent
- **6.8** Payment Orchestration Agent
- **6.9** Demand Forecasting Agent
- **6.10** Supplier Negotiation Agent

---

## Setup and Dependencies

Install required packages before running the examples.

In [ ]:
# Install dependencies
# !pip install langgraph langchain langchain-anthropic langchain-openai pydantic numpy

In [ ]:
# Common imports used across listings
from typing import TypedDict, Optional, List, Dict, Any, Literal
from datetime import datetime
from enum import Enum
from dataclasses import dataclass
from pydantic import BaseModel, Field
import numpy as np
import json
import asyncio

---

## Listing 6.1: Core Retail Agent State Schema

Defines the shared state structure for retail agent workflows, extending LangGraph's TypedDict pattern with retail-specific fields.

In [ ]:
"""
Listing 6.1: Core Retail Agent State Schema

Defines the shared state structure for retail agent workflows.
Extends LangGraph's TypedDict pattern with retail-specific fields.
"""

from typing import TypedDict, Optional, List, Dict, Any
from datetime import datetime
from enum import Enum

class CustomerTier(Enum):
    """Customer loyalty tier classification."""
    STANDARD = "standard"
    SILVER = "silver"
    GOLD = "gold"
    PLATINUM = "platinum"

class TransactionType(Enum):
    """Type of retail transaction."""
    PURCHASE = "purchase"
    RETURN = "return"
    EXCHANGE = "exchange"
    REFUND = "refund"
    PRICE_ADJUSTMENT = "price_adjustment"

class CustomerContext(TypedDict):
    """Customer information and history."""
    customer_id: str
    tier: CustomerTier
    lifetime_value: float
    account_age_days: int
    total_orders: int
    return_rate: float
    last_interaction: datetime
    sentiment_history: List[float]

class TransactionContext(TypedDict):
    """Current transaction details."""
    transaction_id: str
    transaction_type: TransactionType
    order_id: Optional[str]
    items: List[Dict[str, Any]]
    total_amount: float
    payment_method: str
    timestamp: datetime

class RetailAgentState(TypedDict):
    """
    Master state schema for retail agent workflows.

    This state object flows through all nodes in the retail agent graph,
    accumulating information and decisions at each step.
    """
    # Core identifiers
    session_id: str
    workflow_type: str

    # Customer and transaction context
    customer: CustomerContext
    transaction: Optional[TransactionContext]

    # Conversation and reasoning
    messages: List[Dict[str, str]]  # Chat history
    reasoning_trace: List[str]       # Agent's reasoning steps

    # Decision tracking
    risk_score: float               # 0.0 - 1.0
    confidence_score: float         # 0.0 - 1.0
    recommended_action: Optional[str]

    # Tool execution results
    tool_results: Dict[str, Any]

    # Human-in-the-loop
    requires_human_review: bool
    human_review_reason: Optional[str]
    human_decision: Optional[str]

    # Audit and compliance
    audit_log: List[Dict[str, Any]]
    compliance_flags: List[str]

    # Workflow control
    current_step: str
    next_steps: List[str]
    is_complete: bool
    error_state: Optional[str]

---

## Listing 6.2: Agentic RAG for Product Knowledge

Implements self-correcting retrieval with query routing and relevance evaluation for conversational commerce.

In [ ]:
"""
Listing 6.2: Agentic RAG for Retail Product Knowledge

Implements self-correcting retrieval with query routing and
relevance evaluation for conversational commerce.
"""

from typing import List, Dict, Any, Optional
from dataclasses import dataclass
# from langchain_core.documents import Document
# from langchain_openai import ChatOpenAI, OpenAIEmbeddings
# from langchain_community.vectorstores import Pinecone
# from langgraph.graph import StateGraph, END

@dataclass
class ProductQuery:
    """Structured product query with extracted constraints."""
    raw_query: str
    category: Optional[str]
    price_range: Optional[tuple]
    attributes: Dict[str, Any]
    implicit_constraints: List[str]

class AgenticProductRAG:
    """
    Self-correcting RAG system for retail product knowledge.

    Implements query routing, relevance evaluation, and
    automatic query refinement when initial results are poor.
    """

    def __init__(
        self,
        llm,
        product_vectorstore,
        review_vectorstore,
        max_refinement_attempts: int = 3
    ):
        self.llm = llm
        self.product_vs = product_vectorstore
        self.review_vs = review_vectorstore
        self.max_refinements = max_refinement_attempts

    def extract_query_structure(self, raw_query: str) -> ProductQuery:
        """
        Extract structured constraints from natural language query.
        Uses LLM to identify explicit and implicit requirements.
        """
        extraction_prompt = f"""
        Analyze this shopping query and extract:
        1. Product category (if identifiable)
        2. Price range (if mentioned)
        3. Explicit attributes (color, size, brand, etc.)
        4. Implicit constraints (occasion, recipient, style preferences)

        Query: {raw_query}

        Return as JSON with keys: category, price_range, attributes, implicit_constraints
        """

        response = self.llm.invoke(extraction_prompt)
        parsed = self._parse_extraction(response.content)

        return ProductQuery(
            raw_query=raw_query,
            category=parsed.get("category"),
            price_range=parsed.get("price_range"),
            attributes=parsed.get("attributes", {}),
            implicit_constraints=parsed.get("implicit_constraints", [])
        )

    def route_query(self, query: ProductQuery) -> List[str]:
        """
        Determine which knowledge sources to query.
        Routes to product catalog, reviews, or both based on query type.
        """
        sources = ["product_catalog"]  # Always search products

        # Add reviews for quality/experience questions
        quality_signals = ["best", "reliable", "quality", "worth", "recommend"]
        if any(signal in query.raw_query.lower() for signal in quality_signals):
            sources.append("reviews")

        return sources

    def retrieve_and_evaluate(
        self,
        query: ProductQuery,
        sources: List[str]
    ) -> tuple:
        """
        Retrieve from specified sources and evaluate relevance.
        Returns documents and a relevance score (0.0 - 1.0).
        """
        all_docs = []
        enhanced_query = self._build_enhanced_query(query)

        if "product_catalog" in sources:
            product_docs = self.product_vs.similarity_search(
                enhanced_query,
                k=10,
                filter=self._build_filter(query)
            )
            all_docs.extend(product_docs)

        if "reviews" in sources:
            review_docs = self.review_vs.similarity_search(
                enhanced_query,
                k=5
            )
            all_docs.extend(review_docs)

        relevance_score = self._evaluate_relevance(query, all_docs)
        return all_docs, relevance_score

    def _evaluate_relevance(self, query: ProductQuery, documents: list) -> float:
        """Use LLM to evaluate how well documents match the query."""
        if not documents:
            return 0.0

        eval_prompt = f"""
        Rate how well these products match the customer's query.

        Query: {query.raw_query}
        Implicit needs: {query.implicit_constraints}

        Products found:
        {self._format_docs_for_eval(documents[:5])}

        Rate relevance from 0.0 (completely irrelevant) to 1.0 (perfect match).
        Return only a decimal number.
        """

        response = self.llm.invoke(eval_prompt)
        try:
            return float(response.content.strip())
        except ValueError:
            return 0.5

    def search(self, raw_query: str) -> Dict[str, Any]:
        """
        Execute agentic search with self-correction.
        Main entry point for product search.
        """
        query = self.extract_query_structure(raw_query)
        sources = self.route_query(query)

        attempts = 0
        best_docs = []
        best_score = 0.0

        while attempts < self.max_refinements:
            docs, score = self.retrieve_and_evaluate(query, sources)

            if score > best_score:
                best_docs = docs
                best_score = score

            if score >= 0.7:
                break

            attempts += 1
            if attempts < self.max_refinements:
                query = self.refine_query(query, docs)

        return {
            "documents": best_docs,
            "relevance_score": best_score,
            "refinement_attempts": attempts,
            "final_query": query
        }

    def _build_enhanced_query(self, query: ProductQuery) -> str:
        """Build search query from structured components."""
        parts = [query.raw_query]
        if query.category:
            parts.append(f"category:{query.category}")
        for key, value in query.attributes.items():
            parts.append(f"{key}:{value}")
        return " ".join(parts)

    def _build_filter(self, query: ProductQuery) -> Optional[Dict]:
        """Build vector store filter from query constraints."""
        filters = {}
        if query.category:
            filters["category"] = query.category
        if query.price_range:
            filters["price"] = {
                "$gte": query.price_range[0],
                "$lte": query.price_range[1]
            }
        return filters if filters else None

    def _format_docs_for_eval(self, docs: list) -> str:
        """Format documents for LLM evaluation."""
        return "\n".join([
            f"- {doc.metadata.get('name', 'Unknown')}: {doc.page_content[:200]}"
            for doc in docs
        ])

    def _parse_extraction(self, content: str) -> Dict:
        """Parse LLM extraction response."""
        try:
            return json.loads(content)
        except json.JSONDecodeError:
            return {}

---

## Listing 6.3: Hybrid Personalization Engine

Combines collaborative filtering signals with LLM reasoning for explainable, context-aware recommendations.

In [ ]:
"""
Listing 6.3: Hybrid Personalization Engine

Combines collaborative filtering signals with LLM reasoning
for explainable, context-aware recommendations.
"""

from typing import List, Dict, Any, Optional
from dataclasses import dataclass
import numpy as np

@dataclass
class CustomerProfile:
    """Rich customer profile for personalization."""
    customer_id: str
    purchase_history: List[Dict]
    browse_history: List[Dict]
    explicit_preferences: Dict[str, Any]
    inferred_preferences: Dict[str, float]
    segment: str
    lifetime_value: float

@dataclass
class RecommendationResult:
    """Recommendation with explanation."""
    product_id: str
    product_name: str
    score: float
    explanation: str
    reasoning_factors: Dict[str, float]

class HybridPersonalizationEngine:
    """
    Personalization engine combining ML signals with LLM reasoning.

    Produces explainable recommendations that can be presented
    conversationally to customers.
    """

    def __init__(
        self,
        collaborative_model,  # Pre-trained CF model
        content_model,        # Content-based model
        llm,
        product_catalog: Dict[str, Any]
    ):
        self.cf_model = collaborative_model
        self.content_model = content_model
        self.llm = llm
        self.catalog = product_catalog

    def get_recommendations(
        self,
        profile: CustomerProfile,
        context: Dict[str, Any],
        n_recommendations: int = 5
    ) -> List[RecommendationResult]:
        """
        Generate personalized recommendations with explanations.
        
        Combines collaborative filtering, content-based filtering,
        and LLM reasoning for comprehensive recommendations.
        """
        # Get collaborative filtering scores
        cf_scores = self._get_cf_scores(profile.customer_id)
        
        # Get content-based scores
        content_scores = self._get_content_scores(profile)
        
        # Combine scores with weights
        combined_scores = self._combine_scores(cf_scores, content_scores, profile)
        
        # Get top candidates
        top_candidates = sorted(
            combined_scores.items(), 
            key=lambda x: x[1], 
            reverse=True
        )[:n_recommendations * 2]
        
        # Generate explanations using LLM
        recommendations = []
        for product_id, score in top_candidates[:n_recommendations]:
            explanation = self._generate_explanation(
                product_id, profile, context, score
            )
            product = self.catalog.get(product_id, {})
            
            recommendations.append(RecommendationResult(
                product_id=product_id,
                product_name=product.get("name", "Unknown"),
                score=score,
                explanation=explanation,
                reasoning_factors={
                    "collaborative": cf_scores.get(product_id, 0),
                    "content": content_scores.get(product_id, 0),
                    "context": self._context_boost(product_id, context)
                }
            ))
        
        return recommendations

    def _get_cf_scores(self, customer_id: str) -> Dict[str, float]:
        """Get collaborative filtering scores for customer."""
        # In production, this calls the trained CF model
        # Placeholder implementation
        return {}

    def _get_content_scores(self, profile: CustomerProfile) -> Dict[str, float]:
        """Get content-based scores based on profile preferences."""
        # In production, computes similarity to profile preferences
        return {}

    def _combine_scores(
        self,
        cf_scores: Dict[str, float],
        content_scores: Dict[str, float],
        profile: CustomerProfile
    ) -> Dict[str, float]:
        """Combine scores with adaptive weighting."""
        # Weight based on profile characteristics
        cf_weight = 0.6 if profile.lifetime_value > 500 else 0.4
        content_weight = 1 - cf_weight
        
        all_products = set(cf_scores.keys()) | set(content_scores.keys())
        combined = {}
        
        for product_id in all_products:
            cf = cf_scores.get(product_id, 0)
            content = content_scores.get(product_id, 0)
            combined[product_id] = cf * cf_weight + content * content_weight
            
        return combined

    def _context_boost(self, product_id: str, context: Dict[str, Any]) -> float:
        """Calculate context-based score boost."""
        boost = 0.0
        product = self.catalog.get(product_id, {})
        
        # Season/weather boost
        if context.get("season") == product.get("season"):
            boost += 0.1
            
        # Time of day boost
        if context.get("time_of_day") in product.get("best_times", []):
            boost += 0.05
            
        return boost

    def _generate_explanation(
        self,
        product_id: str,
        profile: CustomerProfile,
        context: Dict[str, Any],
        score: float
    ) -> str:
        """Generate natural language explanation for recommendation."""
        product = self.catalog.get(product_id, {})
        
        prompt = f"""
        Generate a brief, conversational explanation for why this product
        is recommended to this customer.

        Product: {product.get('name')} - {product.get('description', '')[:100]}
        
        Customer context:
        - Segment: {profile.segment}
        - Recent purchases: {[p.get('category') for p in profile.purchase_history[-3:]]}
        - Preferences: {profile.explicit_preferences}
        
        Current context:
        - {context}

        Write a 1-2 sentence explanation that feels personal and helpful.
        """
        
        response = self.llm.invoke(prompt)
        return response.content.strip()

---

## Listing 6.6: Intelligent Returns Processing Agent

Autonomous returns processing with policy reasoning, exception handling, and CLV-aware decision making using LangGraph.

In [ ]:
"""
Listing 6.6: Intelligent Returns Processing Agent

Autonomous returns processing with policy reasoning,
exception handling, and CLV-aware decision making.
"""

from typing import TypedDict, List, Optional, Dict, Literal
from dataclasses import dataclass
from enum import Enum
# from langgraph.graph import StateGraph, END
# from langchain_anthropic import ChatAnthropic

class ReturnReason(Enum):
    DEFECTIVE = "defective"
    WRONG_ITEM = "wrong_item"
    NOT_AS_DESCRIBED = "not_as_described"
    CHANGED_MIND = "changed_mind"
    BETTER_PRICE = "better_price"
    TOO_LATE = "too_late"
    OTHER = "other"

class ReturnDecision(Enum):
    FULL_REFUND = "full_refund"
    PARTIAL_REFUND = "partial_refund"
    STORE_CREDIT = "store_credit"
    EXCHANGE = "exchange"
    DENY = "deny"
    ESCALATE = "escalate"

@dataclass
class ReturnRequest:
    """Return request details."""
    order_id: str
    customer_id: str
    items: List[Dict]
    reason: ReturnReason
    days_since_purchase: int
    condition: str
    customer_comment: str

class ReturnsAgentState(TypedDict):
    """State for returns processing workflow."""
    request: ReturnRequest
    customer_tier: str
    customer_ltv: float
    return_history: List[Dict]
    policy_evaluation: Optional[Dict]
    risk_assessment: Optional[Dict]
    clv_analysis: Optional[Dict]
    recommended_decision: Optional[ReturnDecision]
    reasoning: List[str]
    requires_human: bool
    human_override: Optional[str]
    final_decision: Optional[Dict]

class IntelligentReturnsAgent:
    """
    AI agent for intelligent returns processing.
    
    Applies policy reasoning, fraud detection, and CLV optimization
    to make nuanced return decisions.
    """
    
    def __init__(self, policy_config: Dict, clv_thresholds: Dict):
        self.policy = policy_config
        self.clv_thresholds = clv_thresholds
        # self.llm = ChatAnthropic(model="claude-sonnet-4-20250514", temperature=0)
        self.graph = self._build_graph()
    
    def _build_graph(self):
        """Build the returns processing workflow graph."""
        # In production: StateGraph with nodes for each step
        # workflow = StateGraph(ReturnsAgentState)
        # workflow.add_node("evaluate_policy", self._evaluate_policy)
        # workflow.add_node("assess_risk", self._assess_risk)
        # workflow.add_node("analyze_clv", self._analyze_clv)
        # workflow.add_node("make_decision", self._make_decision)
        # workflow.add_node("human_review", self._human_review)
        # workflow.set_entry_point("evaluate_policy")
        # return workflow.compile()
        pass
    
    async def _evaluate_policy(self, state: ReturnsAgentState) -> dict:
        """Evaluate return against standard policies."""
        request = state["request"]
        
        evaluation = {
            "within_return_window": request.days_since_purchase <= self.policy["return_window_days"],
            "eligible_reason": request.reason in self.policy["eligible_reasons"],
            "condition_acceptable": request.condition in self.policy["acceptable_conditions"],
            "policy_compliant": False
        }
        
        # Check overall compliance
        evaluation["policy_compliant"] = all([
            evaluation["within_return_window"],
            evaluation["eligible_reason"],
            evaluation["condition_acceptable"]
        ])
        
        reasoning = []
        if not evaluation["within_return_window"]:
            reasoning.append(f"Return window exceeded: {request.days_since_purchase} days")
        if not evaluation["eligible_reason"]:
            reasoning.append(f"Reason '{request.reason.value}' not in standard eligibility")
        if not evaluation["condition_acceptable"]:
            reasoning.append(f"Item condition '{request.condition}' below acceptable threshold")
        
        return {
            "policy_evaluation": evaluation,
            "reasoning": state["reasoning"] + reasoning
        }
    
    async def _assess_risk(self, state: ReturnsAgentState) -> dict:
        """Assess fraud and abuse risk."""
        request = state["request"]
        history = state["return_history"]
        
        # Calculate risk signals
        recent_returns = len([r for r in history 
                            if r.get("days_ago", 999) < 90])
        total_return_value = sum(r.get("value", 0) for r in history)
        
        risk_score = 0.0
        risk_factors = []
        
        # High return frequency
        if recent_returns > 5:
            risk_score += 0.3
            risk_factors.append("High return frequency")
        
        # Pattern of 'changed mind' returns
        changed_mind_count = len([r for r in history 
                                  if r.get("reason") == "changed_mind"])
        if changed_mind_count > 3:
            risk_score += 0.2
            risk_factors.append("Pattern of changed-mind returns")
        
        # High-value return history
        if total_return_value > 1000:
            risk_score += 0.1
            risk_factors.append("High cumulative return value")
        
        risk_assessment = {
            "score": min(risk_score, 1.0),
            "factors": risk_factors,
            "recommendation": "proceed" if risk_score < 0.5 else "review"
        }
        
        return {
            "risk_assessment": risk_assessment,
            "reasoning": state["reasoning"] + risk_factors
        }
    
    async def _analyze_clv(self, state: ReturnsAgentState) -> dict:
        """Analyze customer lifetime value impact."""
        ltv = state["customer_ltv"]
        tier = state["customer_tier"]
        
        # Determine flexibility based on CLV
        if ltv > self.clv_thresholds["high"]:
            flexibility = "high"
            recommendation = "Approve with maximum flexibility"
        elif ltv > self.clv_thresholds["medium"]:
            flexibility = "medium"
            recommendation = "Approve within extended guidelines"
        else:
            flexibility = "standard"
            recommendation = "Apply standard policy"
        
        clv_analysis = {
            "customer_ltv": ltv,
            "tier": tier,
            "flexibility_level": flexibility,
            "recommendation": recommendation,
            "retention_risk": ltv > 500 and tier in ["gold", "platinum"]
        }
        
        return {
            "clv_analysis": clv_analysis,
            "reasoning": state["reasoning"] + [f"CLV analysis: {recommendation}"]
        }
    
    async def _make_decision(self, state: ReturnsAgentState) -> dict:
        """Make final return decision based on all analysis."""
        policy = state["policy_evaluation"]
        risk = state["risk_assessment"]
        clv = state["clv_analysis"]
        
        # Decision logic
        if policy["policy_compliant"] and risk["score"] < 0.3:
            decision = ReturnDecision.FULL_REFUND
            requires_human = False
        elif clv["flexibility_level"] == "high" and risk["score"] < 0.5:
            decision = ReturnDecision.FULL_REFUND
            requires_human = False
        elif risk["score"] >= 0.7:
            decision = ReturnDecision.ESCALATE
            requires_human = True
        elif not policy["policy_compliant"] and clv["flexibility_level"] == "standard":
            decision = ReturnDecision.DENY
            requires_human = False
        else:
            decision = ReturnDecision.STORE_CREDIT
            requires_human = clv["retention_risk"]
        
        return {
            "recommended_decision": decision,
            "requires_human": requires_human,
            "reasoning": state["reasoning"] + [f"Decision: {decision.value}"]
        }
    
    async def process_return(self, request: ReturnRequest, customer_data: Dict) -> Dict:
        """Process a return request through the agent workflow."""
        initial_state = {
            "request": request,
            "customer_tier": customer_data.get("tier", "standard"),
            "customer_ltv": customer_data.get("ltv", 0),
            "return_history": customer_data.get("return_history", []),
            "policy_evaluation": None,
            "risk_assessment": None,
            "clv_analysis": None,
            "recommended_decision": None,
            "reasoning": [],
            "requires_human": False,
            "human_override": None,
            "final_decision": None
        }
        
        # Execute workflow steps
        state = await self._evaluate_policy(initial_state)
        initial_state.update(state)
        
        state = await self._assess_risk(initial_state)
        initial_state.update(state)
        
        state = await self._analyze_clv(initial_state)
        initial_state.update(state)
        
        state = await self._make_decision(initial_state)
        initial_state.update(state)
        
        return initial_state

---

## Listing 6.7: Fraud Investigation Agent

Multi-signal fraud detection with parallel evidence collection, synthesis, and risk scoring.

In [ ]:
"""
Listing 6.7: Fraud Investigation Agent

Multi-signal fraud detection with parallel evidence collection,
synthesis, and risk scoring.
"""

from typing import TypedDict, List, Optional, Dict
from dataclasses import dataclass
from enum import Enum
import asyncio
# from langchain_anthropic import ChatAnthropic

class FraudRiskLevel(Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"
    CRITICAL = "critical"

@dataclass
class FraudSignal:
    """Individual fraud signal from a collector."""
    source: str
    signal_type: str
    risk_contribution: float  # 0.0 - 1.0
    evidence: Dict
    confidence: float

class FraudInvestigationState(TypedDict):
    """State for fraud investigation workflow."""
    transaction_id: str
    transaction_data: Dict
    customer_data: Dict
    signals: List[FraudSignal]
    synthesized_risk: Optional[Dict]
    recommended_action: Optional[str]
    investigation_notes: List[str]

class FraudInvestigationAgent:
    """
    AI agent for comprehensive fraud investigation.
    
    Collects signals from multiple sources in parallel,
    synthesizes evidence, and recommends actions.
    """
    
    def __init__(self, signal_collectors: Dict, thresholds: Dict):
        self.collectors = signal_collectors
        self.thresholds = thresholds
        # self.llm = ChatAnthropic(model="claude-sonnet-4-20250514", temperature=0)
    
    async def investigate(self, transaction_id: str, transaction: Dict, customer: Dict) -> Dict:
        """
        Conduct full fraud investigation.
        
        Collects signals in parallel, synthesizes risk,
        and generates recommendations.
        """
        state = {
            "transaction_id": transaction_id,
            "transaction_data": transaction,
            "customer_data": customer,
            "signals": [],
            "synthesized_risk": None,
            "recommended_action": None,
            "investigation_notes": []
        }
        
        # Collect signals in parallel
        signals = await self._collect_signals_parallel(state)
        state["signals"] = signals
        
        # Synthesize risk
        risk = await self._synthesize_risk(state)
        state["synthesized_risk"] = risk
        
        # Generate recommendation
        recommendation = await self._generate_recommendation(state)
        state["recommended_action"] = recommendation
        
        return state
    
    async def _collect_signals_parallel(self, state: FraudInvestigationState) -> List[FraudSignal]:
        """Collect fraud signals from all sources in parallel."""
        tasks = [
            self._collect_velocity_signal(state),
            self._collect_device_signal(state),
            self._collect_behavioral_signal(state),
            self._collect_network_signal(state)
        ]
        
        results = await asyncio.gather(*tasks, return_exceptions=True)
        
        signals = []
        for result in results:
            if isinstance(result, FraudSignal):
                signals.append(result)
            elif isinstance(result, Exception):
                # Log but continue with other signals
                state["investigation_notes"].append(f"Signal collection error: {str(result)}")
        
        return signals
    
    async def _collect_velocity_signal(self, state: FraudInvestigationState) -> FraudSignal:
        """Analyze transaction velocity patterns."""
        customer_id = state["customer_data"].get("customer_id")
        amount = state["transaction_data"].get("amount", 0)
        
        # In production, query transaction history
        recent_count = 5  # Placeholder
        recent_total = 500  # Placeholder
        
        risk = 0.0
        evidence = {}
        
        # High velocity check
        if recent_count > 10:
            risk += 0.4
            evidence["high_frequency"] = recent_count
        
        # Unusual amount check
        avg_amount = recent_total / max(recent_count, 1)
        if amount > avg_amount * 3:
            risk += 0.3
            evidence["unusual_amount"] = {"current": amount, "average": avg_amount}
        
        return FraudSignal(
            source="velocity_analyzer",
            signal_type="transaction_velocity",
            risk_contribution=min(risk, 1.0),
            evidence=evidence,
            confidence=0.85
        )
    
    async def _collect_device_signal(self, state: FraudInvestigationState) -> FraudSignal:
        """Analyze device fingerprint and characteristics."""
        device = state["transaction_data"].get("device", {})
        
        risk = 0.0
        evidence = {}
        
        # New device check
        if device.get("is_new", False):
            risk += 0.2
            evidence["new_device"] = True
        
        # VPN/Proxy check
        if device.get("vpn_detected", False):
            risk += 0.3
            evidence["vpn_detected"] = True
        
        # Device mismatch
        if device.get("fingerprint_mismatch", False):
            risk += 0.4
            evidence["fingerprint_mismatch"] = True
        
        return FraudSignal(
            source="device_analyzer",
            signal_type="device_trust",
            risk_contribution=min(risk, 1.0),
            evidence=evidence,
            confidence=0.9
        )
    
    async def _collect_behavioral_signal(self, state: FraudInvestigationState) -> FraudSignal:
        """Analyze behavioral patterns."""
        session = state["transaction_data"].get("session", {})
        
        risk = 0.0
        evidence = {}
        
        # Unusual navigation pattern
        if session.get("pages_viewed", 0) < 2:
            risk += 0.2
            evidence["direct_checkout"] = True
        
        # Very fast checkout
        if session.get("session_duration_seconds", 999) < 30:
            risk += 0.3
            evidence["fast_checkout"] = session.get("session_duration_seconds")
        
        return FraudSignal(
            source="behavioral_analyzer",
            signal_type="user_behavior",
            risk_contribution=min(risk, 1.0),
            evidence=evidence,
            confidence=0.75
        )
    
    async def _collect_network_signal(self, state: FraudInvestigationState) -> FraudSignal:
        """Analyze network and location data."""
        network = state["transaction_data"].get("network", {})
        
        risk = 0.0
        evidence = {}
        
        # High-risk country
        if network.get("country") in ["XX", "YY"]:  # Placeholder
            risk += 0.3
            evidence["high_risk_country"] = network.get("country")
        
        # Location mismatch
        if network.get("location_mismatch", False):
            risk += 0.4
            evidence["location_mismatch"] = True
        
        return FraudSignal(
            source="network_analyzer",
            signal_type="network_trust",
            risk_contribution=min(risk, 1.0),
            evidence=evidence,
            confidence=0.8
        )
    
    async def _synthesize_risk(self, state: FraudInvestigationState) -> Dict:
        """Synthesize all signals into overall risk assessment."""
        signals = state["signals"]
        
        if not signals:
            return {"overall_risk": 0.0, "risk_level": FraudRiskLevel.LOW}
        
        # Weighted average based on confidence
        total_weight = sum(s.confidence for s in signals)
        weighted_risk = sum(s.risk_contribution * s.confidence for s in signals) / total_weight
        
        # Determine risk level
        if weighted_risk < 0.3:
            level = FraudRiskLevel.LOW
        elif weighted_risk < 0.5:
            level = FraudRiskLevel.MEDIUM
        elif weighted_risk < 0.7:
            level = FraudRiskLevel.HIGH
        else:
            level = FraudRiskLevel.CRITICAL
        
        return {
            "overall_risk": weighted_risk,
            "risk_level": level,
            "signal_count": len(signals),
            "contributing_factors": [s.source for s in signals if s.risk_contribution > 0.2]
        }
    
    async def _generate_recommendation(self, state: FraudInvestigationState) -> str:
        """Generate action recommendation based on risk synthesis."""
        risk = state["synthesized_risk"]
        level = risk.get("risk_level", FraudRiskLevel.LOW)
        
        recommendations = {
            FraudRiskLevel.LOW: "approve",
            FraudRiskLevel.MEDIUM: "approve_with_monitoring",
            FraudRiskLevel.HIGH: "manual_review",
            FraudRiskLevel.CRITICAL: "block_and_investigate"
        }
        
        return recommendations.get(level, "manual_review")

---

## Listing 6.9: Demand Forecasting Agent

Contextual demand forecasting combining statistical models with LLM-based adjustment for events, weather, and market conditions.

In [ ]:
"""
Listing 6.9: Demand Forecasting Agent

Contextual demand forecasting combining statistical models
with LLM-based adjustment for events, weather, and market conditions.
"""

from typing import TypedDict, List, Optional, Dict
from dataclasses import dataclass
from datetime import datetime
import numpy as np
# from langchain_anthropic import ChatAnthropic

@dataclass
class DemandForecast:
    """Demand forecast with uncertainty bounds."""
    sku_id: str
    location_id: str
    horizon_days: int
    point_forecast: List[float]
    lower_bound: List[float]
    upper_bound: List[float]
    confidence_score: float
    adjustments_applied: List[str]
    reasoning: str
    anomalies_detected: List[str]

@dataclass
class InventoryRecommendation:
    """Inventory action recommendation."""
    sku_id: str
    location_id: str
    action: str  # REORDER, EXPEDITE, TRANSFER, MARKDOWN, HOLD
    quantity: int
    urgency: str
    reasoning: str
    expected_impact: Dict

class DemandForecastState(TypedDict):
    """State for demand forecasting workflow."""
    sku_id: str
    location_id: str
    historical_data: List[Dict]
    calendar_events: List[Dict]
    weather_forecast: Dict
    competitive_intelligence: Dict
    inventory_status: Dict
    base_forecasts: Optional[Dict]
    contextual_adjustments: Optional[Dict]
    ensemble_forecast: Optional[DemandForecast]
    recommendation: Optional[InventoryRecommendation]

class DemandForecastingAgent:
    """
    AI agent for contextual demand forecasting.
    
    Combines statistical forecasting models with LLM reasoning
    for context-aware predictions and inventory recommendations.
    """
    
    def __init__(self, forecasting_models: Dict, inventory_service):
        self.models = forecasting_models
        self.inventory = inventory_service
        # self.llm = ChatAnthropic(model="claude-sonnet-4-20250514", temperature=0.2)
    
    async def forecast(
        self,
        sku_id: str,
        location_id: str,
        horizon_days: int = 30
    ) -> Dict:
        """
        Generate contextual demand forecast with recommendations.
        """
        # Gather context
        historical = await self._get_historical_data(sku_id, location_id)
        events = await self._get_calendar_events(location_id, horizon_days)
        weather = await self._get_weather_forecast(location_id, horizon_days)
        competitive = await self._get_competitive_intel(sku_id)
        inventory = await self._get_inventory_status(sku_id, location_id)
        
        state = {
            "sku_id": sku_id,
            "location_id": location_id,
            "historical_data": historical,
            "calendar_events": events,
            "weather_forecast": weather,
            "competitive_intelligence": competitive,
            "inventory_status": inventory,
            "base_forecasts": None,
            "contextual_adjustments": None,
            "ensemble_forecast": None,
            "recommendation": None
        }
        
        # Generate base forecasts
        base = await self._generate_base_forecasts(state, horizon_days)
        state["base_forecasts"] = base
        
        # Apply contextual adjustments
        adjustments = await self._generate_contextual_adjustments(state)
        state["contextual_adjustments"] = adjustments
        
        # Create ensemble forecast
        ensemble = await self._create_ensemble(state, horizon_days)
        state["ensemble_forecast"] = ensemble
        
        # Analyze for anomalies
        state = await self._analyze_anomalies(state)
        
        # Generate recommendation
        rec = await self._generate_recommendation(state)
        state["recommendation"] = rec
        
        return state
    
    async def _get_historical_data(self, sku_id: str, location_id: str) -> List[Dict]:
        """Fetch historical sales data."""
        # In production, query data warehouse
        return []
    
    async def _get_calendar_events(self, location_id: str, horizon: int) -> List[Dict]:
        """Fetch relevant calendar events."""
        return []
    
    async def _get_weather_forecast(self, location_id: str, horizon: int) -> Dict:
        """Fetch weather forecast."""
        return {}
    
    async def _get_competitive_intel(self, sku_id: str) -> Dict:
        """Fetch competitive intelligence."""
        return {}
    
    async def _get_inventory_status(self, sku_id: str, location_id: str) -> Dict:
        """Fetch current inventory status."""
        return {}
    
    async def _generate_base_forecasts(self, state: DemandForecastState, horizon: int) -> Dict:
        """Generate forecasts from statistical models."""
        historical = state["historical_data"]
        
        forecasts = {}
        
        # In production, call actual forecasting models
        # Placeholder with simple moving average
        if historical:
            values = [d.get("quantity", 0) for d in historical[-30:]]
            avg = np.mean(values) if values else 10
            forecasts["moving_average"] = [avg] * horizon
            forecasts["exponential_smoothing"] = [avg * 1.02] * horizon
            forecasts["arima"] = [avg * 0.98] * horizon
        else:
            forecasts["moving_average"] = [10] * horizon
            forecasts["exponential_smoothing"] = [10] * horizon
            forecasts["arima"] = [10] * horizon
        
        return forecasts
    
    async def _generate_contextual_adjustments(self, state: DemandForecastState) -> Dict:
        """Use LLM to generate contextual adjustments."""
        events = state["calendar_events"]
        weather = state["weather_forecast"]
        competitive = state["competitive_intelligence"]
        
        # Placeholder for LLM-based adjustment logic
        adjustments = {
            "adjustments": [],
            "confidence_impact": 0.0,
            "reasoning": "No significant contextual factors identified"
        }
        
        # Example: holiday adjustment
        for event in events:
            if event.get("type") == "holiday":
                adjustments["adjustments"].append({
                    "days": list(range(event.get("day_offset", 0), event.get("day_offset", 0) + 3)),
                    "multiplier": 1.5,
                    "reason": f"Holiday: {event.get('name')}"
                })
        
        return adjustments
    
    async def _create_ensemble(self, state: DemandForecastState, horizon: int) -> DemandForecast:
        """Create ensemble forecast from base models and adjustments."""
        forecasts = state["base_forecasts"]
        adjustments = state["contextual_adjustments"]
        historical = state["historical_data"]
        
        # Calculate model weights
        weights = self._calculate_model_weights(forecasts, historical)
        
        # Weighted ensemble
        ensemble = self._weighted_ensemble(forecasts, weights)
        
        # Apply adjustments
        adjusted = self._apply_adjustments(ensemble, adjustments.get("adjustments", []))
        
        # Calculate intervals
        confidence_impact = adjustments.get("confidence_impact", 0)
        lower, upper = self._calculate_intervals(adjusted, historical, confidence_impact)
        
        return DemandForecast(
            sku_id=state["sku_id"],
            location_id=state["location_id"],
            horizon_days=horizon,
            point_forecast=adjusted,
            lower_bound=lower,
            upper_bound=upper,
            confidence_score=0.8 - confidence_impact,
            adjustments_applied=[a.get("reason", "") for a in adjustments.get("adjustments", [])],
            reasoning=adjustments.get("reasoning", ""),
            anomalies_detected=[]
        )
    
    async def _analyze_anomalies(self, state: DemandForecastState) -> dict:
        """Analyze forecast for anomalies."""
        forecast = state["ensemble_forecast"]
        historical = state["historical_data"]
        
        anomalies = []
        
        if historical:
            hist_values = [d.get("quantity", 0) for d in historical[-90:]]
            hist_mean = np.mean(hist_values) if hist_values else 0
            hist_std = np.std(hist_values) if hist_values else 1
            
            # Flag days with forecast > 3 sigma from historical
            for i, value in enumerate(forecast.point_forecast):
                if hist_std > 0 and abs(value - hist_mean) > 3 * hist_std:
                    anomalies.append(
                        f"Day {i}: Forecast {value:.0f} is {abs(value-hist_mean)/hist_std:.1f} "
                        f"std devs from historical mean"
                    )
        
        forecast.anomalies_detected = anomalies
        return {"ensemble_forecast": forecast}
    
    async def _generate_recommendation(self, state: DemandForecastState) -> InventoryRecommendation:
        """Generate inventory optimization recommendation."""
        forecast = state["ensemble_forecast"]
        inventory = state["inventory_status"]
        
        current_stock = inventory.get("on_hand", 0)
        on_order = inventory.get("on_order", 0)
        lead_time_days = inventory.get("lead_time_days", 7)
        
        # Calculate expected demand during lead time
        lead_time_demand = sum(forecast.point_forecast[:lead_time_days])
        
        # Determine position
        position = current_stock + on_order
        days_of_supply = position / max(1, np.mean(forecast.point_forecast))
        
        # Simple decision logic
        if days_of_supply < lead_time_days:
            action = "EXPEDITE"
            urgency = "immediate"
        elif days_of_supply < lead_time_days * 1.5:
            action = "REORDER"
            urgency = "planned"
        elif days_of_supply > 60:
            action = "MARKDOWN"
            urgency = "opportunistic"
        else:
            action = "HOLD"
            urgency = "none"
        
        quantity = max(0, int(lead_time_demand * 1.2 - position))
        
        return InventoryRecommendation(
            sku_id=state["sku_id"],
            location_id=state["location_id"],
            action=action,
            quantity=quantity,
            urgency=urgency,
            reasoning=f"Days of supply: {days_of_supply:.1f}, Lead time: {lead_time_days}",
            expected_impact={"stockout_risk_reduction": 0.3}
        )
    
    def _calculate_model_weights(self, forecasts: Dict, historical: List) -> Dict:
        """Calculate weights based on recent forecast accuracy."""
        n = len(forecasts)
        return {k: 1/n for k in forecasts.keys()}
    
    def _weighted_ensemble(self, forecasts: Dict, weights: Dict) -> List[float]:
        """Create weighted ensemble forecast."""
        arrays = list(forecasts.values())
        weight_values = [weights[k] for k in forecasts.keys()]
        return list(np.average(arrays, axis=0, weights=weight_values))
    
    def _apply_adjustments(self, forecast: List[float], adjustments: List[Dict]) -> List[float]:
        """Apply adjustment factors to forecast."""
        result = forecast.copy()
        for adj in adjustments:
            for day in adj.get("days", []):
                if 0 <= day < len(result):
                    result[day] *= adj.get("multiplier", 1.0)
        return result
    
    def _calculate_intervals(self, forecast: List[float], historical: List, confidence_impact: float) -> tuple:
        """Calculate prediction intervals."""
        if historical:
            hist_values = [d.get("quantity", 0) for d in historical[-30:]]
            std = np.std(hist_values) if hist_values else 5
        else:
            std = 5
        
        interval_width = 1.645 * std * (1 - confidence_impact)
        lower = [max(0, f - interval_width) for f in forecast]
        upper = [f + interval_width for f in forecast]
        
        return lower, upper

---

## Listing 6.10: Supplier Negotiation Agent

Autonomous supplier negotiation with relationship context, strategy selection, and multi-objective optimization.

In [ ]:
"""
Listing 6.10: Supplier Negotiation Agent

Autonomous supplier negotiation with relationship context,
strategy selection, and multi-objective optimization.
"""

from typing import TypedDict, List, Optional, Dict
from dataclasses import dataclass
from enum import Enum
# from langchain_anthropic import ChatAnthropic

class NegotiationStrategy(Enum):
    COLLABORATIVE = "collaborative"  # Long-term partnership focus
    COMPETITIVE = "competitive"      # Price optimization focus
    BALANCED = "balanced"            # Multi-objective optimization
    RELATIONSHIP_REPAIR = "relationship_repair"  # Address past issues

@dataclass
class SupplierContext:
    """Comprehensive supplier relationship context."""
    supplier_id: str
    supplier_name: str
    relationship_tenure_years: float
    total_spend_ytd: float
    quality_score: float  # 0-100
    delivery_score: float  # 0-100
    responsiveness_score: float  # 0-100
    strategic_importance: str  # critical, important, standard
    recent_issues: List[Dict]
    contract_expiry: str
    alternative_suppliers: List[Dict]

@dataclass
class NegotiationOutcome:
    """Result of negotiation interaction."""
    success: bool
    agreed_terms: Dict
    concessions_made: List[str]
    concessions_received: List[str]
    relationship_impact: str  # improved, maintained, strained
    follow_up_actions: List[str]
    reasoning: str

class SupplierNegotiationState(TypedDict):
    """State for supplier negotiation workflow."""
    request_type: str  # price_negotiation, delivery_expedite, quality_issue
    supplier_context: SupplierContext
    our_position: Dict
    supplier_position: Optional[Dict]
    strategy: Optional[NegotiationStrategy]
    negotiation_history: List[Dict]
    current_offer: Optional[Dict]
    authority_limits: Dict
    outcome: Optional[NegotiationOutcome]

class SupplierNegotiationAgent:
    """
    AI agent for supplier negotiations.

    Conducts autonomous negotiations within defined authority,
    balancing transactional value with relationship health.
    """

    def __init__(self, supplier_service, authority_config: Dict):
        self.supplier_service = supplier_service
        self.authority_config = authority_config
        # self.llm = ChatAnthropic(model="claude-sonnet-4-20250514", temperature=0.3)

    async def negotiate(
        self,
        request_type: str,
        supplier_id: str,
        our_requirements: Dict,
        authority_limits: Dict
    ) -> NegotiationOutcome:
        """
        Conduct negotiation with supplier.

        Selects appropriate strategy, manages multi-round interactions,
        and optimizes for both transaction and relationship outcomes.
        """
        # Gather supplier context
        context = await self.supplier_service.get_context(supplier_id)

        # Select negotiation strategy
        strategy = await self._select_strategy(request_type, context)

        # Initialize state
        state = {
            "request_type": request_type,
            "supplier_context": context,
            "our_position": our_requirements,
            "supplier_position": None,
            "strategy": strategy,
            "negotiation_history": [],
            "current_offer": None,
            "authority_limits": authority_limits,
            "outcome": None
        }

        # Conduct negotiation rounds
        max_rounds = 5
        for round_num in range(max_rounds):
            state = await self._conduct_round(state, round_num)

            if state["outcome"] is not None:
                break

        # If no resolution, generate final outcome
        if state["outcome"] is None:
            state = await self._generate_no_agreement_outcome(state)

        return state["outcome"]

    async def _select_strategy(
        self,
        request_type: str,
        context: SupplierContext
    ) -> NegotiationStrategy:
        """Select negotiation strategy based on context."""
        # Strategy selection logic
        if context.strategic_importance == "critical":
            if context.recent_issues:
                return NegotiationStrategy.RELATIONSHIP_REPAIR
            return NegotiationStrategy.COLLABORATIVE
        
        if len(context.alternative_suppliers) > 3 and context.strategic_importance == "standard":
            return NegotiationStrategy.COMPETITIVE
        
        return NegotiationStrategy.BALANCED

    async def _conduct_round(
        self,
        state: SupplierNegotiationState,
        round_num: int
    ) -> SupplierNegotiationState:
        """Conduct one round of negotiation."""
        strategy = state["strategy"]
        context = state["supplier_context"]
        our_position = state["our_position"]
        history = state["negotiation_history"]
        limits = state["authority_limits"]

        # Generate our offer based on strategy
        our_move = await self._generate_offer(state, round_num)

        # Record our move
        history.append({
            "round": round_num,
            "party": "us",
            "offer": our_move.get("offer"),
            "message": our_move.get("message")
        })

        # Simulate supplier response
        supplier_response = await self._simulate_supplier_response(state, our_move, round_num)

        # Record supplier response
        history.append({
            "round": round_num,
            "party": "supplier",
            "offer": supplier_response.get("counter_offer"),
            "message": supplier_response.get("message")
        })

        # Check for agreement
        if supplier_response.get("accepted"):
            state["outcome"] = NegotiationOutcome(
                success=True,
                agreed_terms=our_move.get("offer"),
                concessions_made=our_move.get("concessions_offered", []),
                concessions_received=our_move.get("concessions_requested", []),
                relationship_impact="maintained",
                follow_up_actions=["Send confirmation", "Update contract records"],
                reasoning=f"Agreement reached in round {round_num + 1}"
            )
        elif our_move.get("acceptable_if_agreed") and self._within_limits(
            supplier_response.get("counter_offer"), limits
        ):
            # Evaluate if we should accept their counter
            if self._should_accept(state, supplier_response.get("counter_offer")):
                state["outcome"] = NegotiationOutcome(
                    success=True,
                    agreed_terms=supplier_response.get("counter_offer"),
                    concessions_made=["Accepted counter-offer"],
                    concessions_received=[],
                    relationship_impact="maintained",
                    follow_up_actions=["Send acceptance", "Document terms"],
                    reasoning="Counter-offer within acceptable limits"
                )

        state["supplier_position"] = supplier_response.get("counter_offer")
        state["negotiation_history"] = history

        return state

    async def _generate_offer(self, state: SupplierNegotiationState, round_num: int) -> Dict:
        """Generate negotiation offer based on strategy and round."""
        strategy = state["strategy"]
        our_position = state["our_position"]
        supplier_position = state["supplier_position"]
        
        # Starting offer
        if round_num == 0:
            return {
                "offer": our_position,
                "message": "Initial proposal based on current market conditions",
                "concessions_offered": [],
                "concessions_requested": [],
                "acceptable_if_agreed": True
            }
        
        # Subsequent rounds - make concessions based on strategy
        concession_rate = {
            NegotiationStrategy.COLLABORATIVE: 0.8,
            NegotiationStrategy.COMPETITIVE: 0.5,
            NegotiationStrategy.BALANCED: 0.65,
            NegotiationStrategy.RELATIONSHIP_REPAIR: 0.9
        }.get(strategy, 0.65)
        
        # Simple concession logic
        if supplier_position and "price" in supplier_position:
            new_price = our_position.get("price", 0) * (1 + (1 - concession_rate) * 0.1)
            return {
                "offer": {"price": new_price},
                "message": f"Revised offer considering your position",
                "concessions_offered": [f"Increased price offer by {(1-concession_rate)*10:.1f}%"],
                "concessions_requested": [],
                "acceptable_if_agreed": True
            }
        
        return {
            "offer": our_position,
            "message": "Maintaining position",
            "concessions_offered": [],
            "concessions_requested": [],
            "acceptable_if_agreed": False
        }

    async def _simulate_supplier_response(
        self,
        state: SupplierNegotiationState,
        our_move: Dict,
        round_num: int
    ) -> Dict:
        """Simulate supplier response for demonstration."""
        # In production, this would be actual supplier communication
        context = state["supplier_context"]
        
        # Simple simulation logic
        accept_probability = 0.2 + (round_num * 0.15)
        
        if context.strategic_importance == "critical":
            accept_probability += 0.2
        
        import random
        if random.random() < accept_probability:
            return {
                "accepted": True,
                "counter_offer": None,
                "message": "We accept your proposal"
            }
        
        # Generate counter-offer
        our_price = our_move.get("offer", {}).get("price", 100)
        counter_price = our_price * 1.1  # 10% higher
        
        return {
            "accepted": False,
            "counter_offer": {"price": counter_price},
            "message": "We need better terms"
        }

    async def _generate_no_agreement_outcome(
        self,
        state: SupplierNegotiationState
    ) -> SupplierNegotiationState:
        """Generate outcome when no agreement is reached."""
        state["outcome"] = NegotiationOutcome(
            success=False,
            agreed_terms={},
            concessions_made=[],
            concessions_received=[],
            relationship_impact="strained",
            follow_up_actions=[
                "Escalate to procurement manager",
                "Evaluate alternative suppliers",
                "Document negotiation history"
            ],
            reasoning="Maximum negotiation rounds reached without agreement"
        )
        return state

    def _within_limits(self, offer: Dict, limits: Dict) -> bool:
        """Check if offer is within authority limits."""
        if not offer:
            return False

        if "price" in offer and "max_price" in limits:
            if offer["price"] > limits["max_price"]:
                return False

        return True

    def _should_accept(self, state: SupplierNegotiationState, counter_offer: Dict) -> bool:
        """Determine if we should accept the counter-offer."""
        limits = state["authority_limits"]
        
        if not counter_offer:
            return False
        
        # Simple acceptance logic
        if "price" in counter_offer and "max_price" in limits:
            return counter_offer["price"] <= limits["max_price"]
        
        return True

---

## Summary

This notebook contains the complete code implementations for Chapter 6: AI Agents in Retail and E-Commerce. The listings demonstrate:

**Customer-Facing Agents:**
- Core state schemas for retail workflows (Listing 6.1)
- Agentic RAG with self-correction for product search (Listing 6.2)
- Hybrid personalization combining ML and LLM reasoning (Listing 6.3)

**Transaction Processing:**
- Intelligent returns processing with CLV awareness (Listing 6.6)
- Multi-signal fraud investigation (Listing 6.7)

**Supply Chain Intelligence:**
- Contextual demand forecasting with ensemble models (Listing 6.9)
- Autonomous supplier negotiation (Listing 6.10)

### Key Patterns Demonstrated

1. **TypedDict State Schemas**: All agents use strongly-typed state for LangGraph integration
2. **Async/Await Patterns**: Parallel signal collection and non-blocking operations
3. **LLM Reasoning Integration**: Combining statistical models with contextual LLM analysis
4. **Human-in-the-Loop**: Authority limits and escalation triggers for high-stakes decisions
5. **Multi-Agent Coordination**: State passing between specialized agents

### Production Considerations

To deploy these agents in production:

1. **Replace placeholder integrations** with actual database queries, API calls, and ML model inference
2. **Add comprehensive logging** for audit trails and debugging
3. **Implement rate limiting** for LLM API calls
4. **Configure model fallbacks** for resilience
5. **Set up monitoring** for latency, error rates, and decision quality metrics

---

*For more details, see Chapter 6 of "Practical AI Agents: A Comprehensive Guide to Designing, Building, and Deploying Autonomous LLM-Powered Systems"*